In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://nesfb.com/Interest_Rates"

date1 = (date.today()).strftime('%d%m%Y') # Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' # Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find("table",class_="table table-bordered")
    rows=table.find_all("tr")

    list_of_interest_rates = []
    for i in rows[4:]:  #Skip heading
        data=i.find_all("td")
        #print(data)
        row=[tr.text for tr in data]
        list_of_interest_rates.append(row)

    list_of_interest_rates=[sublist[:3] + sublist[5:] for sublist in list_of_interest_rates]
    #print(list_of_interest_rates)

    final_rates = []
    tax_saver_row =[]
    row_count = 0
    for i in list_of_interest_rates:
        x = ['Northeast SF Bank']
        x.extend(i)  # Tenure, gen_rate, sr_rate
        x.insert(3, '')
        x.extend([''])
        if row_count == 9:
            tax_saver_row = ['Northeast SF Bank', 'Tax-saver Fixed Deposit', x[2], x[3], x[4], x[5]]
            final_rates.append(tax_saver_row)
        final_rates.append(x)
        row_count += 1
    #print(final_rates)

    pd.set_option('display.max_columns', None)
    int_rate_df=pd.DataFrame(final_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate','sr_rate',
                                        'annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nNortheast SF Bank = Success. Number of rows added = ", len(final_rates))
finally:
    # Close the driver
    driver.quit()

            bank_name                                             tenure  \
0   Northeast SF Bank                                        7 - 14 Days   
1   Northeast SF Bank                                       15 - 29 Days   
2   Northeast SF Bank              30 - 45 Days(1 month-1 month 15 days)   
3   Northeast SF Bank             46 - 90 Days(1 month 16 days-3 months)   
4   Northeast SF Bank             91 - 180 Days(3 months 1 day-6 months)   
5   Northeast SF Bank              181 - 365 Days(6 months 1 day-1 year)   
6   Northeast SF Bank       366 - 545 Days(1 year 1 day-1 year 6 months)   
7   Northeast SF Bank  546 - 1111 Days(1 year 6 months 1day-3 years 1...   
8   Northeast SF Bank  1112 - 1825 Days(3 years 17 days-4 years 11 mo...   
9   Northeast SF Bank                            Tax-saver Fixed Deposit   
10  Northeast SF Bank                1826 - 3650 Days(5 years -10 years)   

   gen_rate annualised_gen_rate sr_rate annualised_sr_rate  
0     3.25%               